In [ ]:
import plotly.express as px
import pandas as pd

In [ ]:
dataset = '/content/flights_cleaned.csv'
df = pd.read_csv(dataset)
df.head()

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,date,Day_of_Week,Is_weekend,departure_period,Departure_Delay_Flag,Arrival_Delay_Flag,Delay_Severity,Route,Distance_Category,delay_cause
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,01-01-2013,1,0,Morning,1,1,Slight Delay,EWR-IAH,Long Distance,Minor Operational Delay
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,01-01-2013,1,0,Morning,1,1,Slight Delay,LGA-IAH,Long Distance,Ground Operations Delay
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,01-01-2013,1,0,Morning,1,1,Medium Delay,JFK-MIA,Long Distance,Ground Operations Delay
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,...,01-01-2013,1,0,Morning,0,0,On Time,JFK-BQN,Long Distance,On-time
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,...,01-01-2013,1,0,Morning,0,0,On Time,LGA-ATL,Medium Distance,On-time


In [ ]:
top_10_airlines = df['name'].value_counts().head(10)
fig = px.bar(
    x=top_10_airlines.index,
    y=top_10_airlines.values,
    color=top_10_airlines.index,
    title='Top 10 Airlines by Flight Count',
    labels={'x': 'Airline Name', 'y': 'Number of Flights'}
)
fig.show()

In [ ]:
top_5_routes = df['Route'].value_counts().head(5)

fig = px.bar(
    x=top_5_routes.index,
    y=top_5_routes.values,
    color=top_5_routes.index,
    title='Top 5 Flight Routes by Count',
    labels={'x': 'Route', 'y': 'Number of Flights'}
)
fig.show()

In [ ]:
flights_by_month = df['month'].value_counts().sort_index()

month_names = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}
flights_by_month.index = flights_by_month.index.map(month_names)

fig = px.line(
    x=flights_by_month.index,
    y=flights_by_month.values,
    title='Number of Flights by Month',
    labels={'x': 'Month', 'y': 'Number of Flights'},
    category_orders={'x': list(month_names.values())},
    markers=True
)
fig.update_traces(mode='lines+markers')
fig.show()

In [ ]:
heatmap_data = df.groupby(['Day_of_Week', 'departure_period']).size().unstack(fill_value=0)

day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
heatmap_data = heatmap_data.rename(index=day_names)

departure_period_order = ['Night', 'Morning', 'Afternoon', 'Evening', 'Late Night']
heatmap_data = heatmap_data.reindex(columns=departure_period_order, fill_value=0)

fig = px.imshow(
    heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale="Plasma",
    title='Flight Distribution by Day of Week and Departure Period',
    labels={'x': 'Departure Period', 'y': 'Day of Week', 'color': 'Number of Flights'}
)

fig.update_xaxes(side="top")
fig.show()

In [ ]:
origin_counts = df['origin'].value_counts()
display(origin_counts)

,count
origin,
EWR,117127
JFK,109079
LGA,101140


In [ ]:

origin_airports = df['origin'].value_counts().index.tolist()

df_origins = df[df['origin'].isin(origin_airports)].copy()

flights_by_origin_month = df_origins.groupby(['month', 'origin']).size().reset_index(name='count')

month_names = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}
flights_by_origin_month['month_name'] = flights_by_origin_month['month'].map(month_names)

fig = px.line(
    flights_by_origin_month,
    x='month_name',
    y='count',
    color='origin',
    title='Flight Distribution by Three Origin Airports Over Month',
    labels={'month_name': 'Month', 'count': 'Number of Flights', 'origin': 'Origin Airport'},
    category_orders={'month_name': list(month_names.values())},
    markers=True
)
fig.update_traces(mode='lines+markers')
fig.show()

In [ ]:
delay_causes_by_airline = df.groupby(['name', 'delay_cause']).size().reset_index(name='flight_count')
delay_causes_by_airline.head()

,name,delay_cause,flight_count
0,AirTran Airways Corporation,Air Traffic Control Delay,252
1,AirTran Airways Corporation,Aircraft / Maintenance Issue,157
2,AirTran Airways Corporation,Ground Operations Delay,582
3,AirTran Airways Corporation,Minor Operational Delay,830
4,AirTran Airways Corporation,On-time,1280


In [ ]:
delay_causes_by_airline_filtered = delay_causes_by_airline[delay_causes_by_airline['delay_cause'] != 'On-time']

heatmap_data_airlines = delay_causes_by_airline_filtered.pivot_table(
    index='name',
    columns='delay_cause',
    values='flight_count',
    fill_value=0
)

fig_heatmap = px.imshow(
    heatmap_data_airlines,
    labels=dict(x="Delay Cause", y="Airline Name", color="Number of Flights"),
    x=heatmap_data_airlines.columns,
    y=heatmap_data_airlines.index,
    color_continuous_scale="Viridis",
    title="Heatmap of Delay Causes by Airline",
    height=800,
    width=1800
)

fig_heatmap.update_xaxes(side="top")
fig_heatmap.show()

In [ ]:
specific_delay_causes = ['Carrier Delay', 'Weather / Major Disruption', 'National Air System Delay']
specific_delays_df = df[df['delay_cause'].isin(specific_delay_causes)]
specific_delays_df.head()
display(specific_delays_df.shape)

(3843, 32)

In [ ]:
specific_delays_by_origin = specific_delays_df.groupby(['origin', 'delay_cause']).size().reset_index(name='flight_count')

fig = px.bar(
    specific_delays_by_origin,
    x='origin',
    y='flight_count',
    color='delay_cause',
    title='Distribution of Specific Delay Types by Origin Airport',
    labels={'origin': 'Origin Airport', 'flight_count': 'Number of Flights', 'delay_cause': 'Delay Cause'},
    barmode='group'
)
fig.show()

In [ ]:
def convert_to_hour(time_val):
    if pd.isna(time_val):
        return None
    time_str = str(int(time_val)).zfill(4)
    if len(time_str) < 4:

        time_str = time_str.rjust(4, '0')
    try:
        return int(time_str[:2])
    except ValueError:
        return None


df['departure_hour'] = df['dep_time'].apply(convert_to_hour)

delayed_flights_df = df[df['dep_delay'] > 0].copy()

delayed_flights_hourly = delayed_flights_df['departure_hour'].value_counts().sort_index().reset_index()
delayed_flights_hourly.columns = ['departure_hour', 'flight_count']

fig = px.line(
    delayed_flights_hourly,
    x='departure_hour',
    y='flight_count',
    title='Frequency of Delayed Flights by Departure Hour',
    labels={'departure_hour': 'Departure Hour', 'flight_count': 'Number of Delayed Flights'},
    markers=True
)
fig.update_traces(mode='lines+markers')
fig.show()